In [1]:
%pip install -qU langchain langchain-openai langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader

documento = PyPDFLoader("documentos/Regras_26_27_PT_BR_52925fd6d0.pdf").load()

C:\Users\IATR\AppData\Local\Temp\ipykernel_19420\4086271058.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
)

chunks = splitter.split_documents(documento)

In [11]:
chunks[80].page_content

'50\n5. Área de meta\n São traçadas duas linhas perpendiculares à linha de fundo, a 5,5 m de distância do \ninterior de cada trave. Essas linhas se prolongam para o interior do campo de jogo \npor 5,5 m e são unidas por uma linha paralela à linha de fundo. A área delimitada \npor essas linhas e pela linha de fundo constitui a área de meta.\n6. A área penal\n São traçadas duas linhas perpendiculares à linha de fundo, a 16,5 m de distância \ndo interior de cada trave. Essas linhas se prolongam para o interior do campo de \njogo por 16,5 m e são unidas por uma linha paralela à linha de fundo. A área \ndelimitada por essas linhas e pela linha de fundo constitui a área penal.\n Em cada área penal, a marca penal é marcada a 11 m de distância do ponto médio \nentre as traves.\n Um arco de círculo com um raio de 9,15 m a partir do centro de cada marca penal é \ntraçado no exterior da área penal.\n7. A área do escanteio\n A área do escanteio ou área de tiro de canto é delimitada por um quarto d

In [14]:
from openai import OpenAI
from os import getenv

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getenv("OPENROUTER_API_KEY")
)

chunk_texts = [chunk.page_content for chunk in chunks]
batch_size = 256
all_embeddings = []

for start in range(0, len(chunk_texts), batch_size):
    batch = chunk_texts[start:start + batch_size]
    response = client.embeddings.create(
        model="nvidia/nemotron-3-embed-1b:free",
        input=batch,
        encoding_format="float",
    )
    all_embeddings.extend(response.data)

print(f"Created {len(all_embeddings)} embeddings")
print(all_embeddings[0].embedding[:10])

Created 449 embeddings
[0.11317405, 0.01420262, -0.017303111, -0.02345115, 0.0027289246, -0.005477337, -0.009875056, 0.027367847, 0.04879975, -0.0044496865]


In [18]:
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import Embeddings

class OpenRouterEmbeddings(Embeddings):
    def embed_documents(self, texts):
        texts = [text for text in texts if isinstance(text, str) and text.strip()]
        if not texts:
            return []

        response = client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=texts,
            encoding_format="float",
        )
        data = getattr(response, "data", None) or []
        return [item.embedding for item in data if getattr(item, "embedding", None) is not None]

    def embed_query(self, text):
        response = client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=[text],
            encoding_format="float",
        )
        data = getattr(response, "data", None) or []
        if not data:
            return []
        return data[0].embedding

embeddings = OpenRouterEmbeddings()

valid_chunks = [chunk for chunk in chunks if getattr(chunk, "page_content", "").strip()]
vectorstore = InMemoryVectorStore.from_documents(
    documents=valid_chunks,
    embedding=embeddings
)

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [21]:
retriever.invoke("Qual é a diferença entre a regra 26 e a regra 27?")

[]

In [ ]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="google/gemma-4-26b-a4b-it:free",
    temperature=0.8,
)

response = model.invoke("")
print(response.content)

Justin Bieber was born on March 1, 1994.

The Super Bowl played in 1994 (Super Bowl XXVIII, which took place in January 1994) was won by the **Dallas Cowboys**, who defeated the Buffalo Bills.
